# 6. 模型格式与序列化

高效的模型存储格式直接影响加载速度、内存占用和跨平台兼容性。不同格式针对不同场景优化。

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import json
import struct
import os
import time

torch.manual_seed(42)
print(f"PyTorch version: {torch.__version__}")

### GGUF格式详解

GGUF是llama.cpp的标准格式，支持元数据嵌入、多种量化类型和mmap加载。

In [ ]:
class GGUFFileSimulator:
    """GGUF文件格式模拟"""
    GGUF_MAGIC = 0x46475547
    TYPES = {0: 'F32', 1: 'F16', 2: 'Q4_0', 3: 'Q4_1', 6: 'Q5_0', 7: 'Q5_1',
             8: 'Q8_0', 12: 'Q4_K', 13: 'Q5_K', 14: 'Q6_K'}

    def __init__(self):
        self.metadata = {}
        self.tensor_infos = {}
        self.alignment = 32

    def add_metadata(self, key, value):
        self.metadata[key] = value

    def add_tensor_info(self, name, shape, dtype='Q4_K', offset=0):
        n_elements = 1
        for s in shape:
            n_elements *= s
        bytes_per_element = {'F32': 4, 'F16': 2, 'Q4_0': 0.5625, 'Q4_1': 0.625,
                             'Q5_0': 0.6875, 'Q5_1': 0.75, 'Q8_0': 1.0625,
                             'Q4_K': 0.5625, 'Q5_K': 0.6875, 'Q6_K': 0.8125}
        size_bytes = int(n_elements * bytes_per_element.get(dtype, 2))
        self.tensor_infos[name] = {
            'shape': shape, 'dtype': dtype, 'offset': offset, 'size': size_bytes
        }

    def estimate_file_size(self):
        header_size = 64
        metadata_size = sum(len(k) + 16 for k in self.metadata)
        tensor_info_size = sum(len(k) + 64 for k in self.tensor_infos)
        tensor_data_size = sum(t['size'] for t in self.tensor_infos.values())
        return header_size + metadata_size + tensor_info_size + tensor_data_size

gguf = GGUFFileSimulator()
gguf.add_metadata('general.architecture', 'llama')
gguf.add_metadata('llm.context_length', 4096)
gguf.add_metadata('llm.embedding_length', 4096)
gguf.add_metadata('llm.layer_count', 32)
gguf.add_metadata('llm.attention.head_count', 32)

gguf.add_tensor_info(f'token_embd.weight', [4096, 32000], 'Q4_K')
for i in range(32):
    gguf.add_tensor_info(f'blk.{i}.attn_q.weight', [4096, 4096], 'Q4_K')
    gguf.add_tensor_info(f'blk.{i}.attn_k.weight', [4096, 512], 'Q4_K')
    gguf.add_tensor_info(f'blk.{i}.attn_v.weight', [4096, 512], 'Q4_K')
    gguf.add_tensor_info(f'blk.{i}.attn_output.weight', [4096, 4096], 'Q4_K')
    gguf.add_tensor_info(f'blk.{i}.ffn_gate.weight', [4096, 11008], 'Q4_K')
    gguf.add_tensor_info(f'blk.{i}.ffn_up.weight', [4096, 11008], 'Q4_K')
    gguf.add_tensor_info(f'blk.{i}.ffn_down.weight', [11008, 4096], 'Q4_K')

print(f"=== GGUF格式模拟 (7B模型) ===")
print(f"元数据条目: {len(gguf.metadata)}")
print(f"张量条目: {len(gguf.tensor_infos)}")
print(f"\n元数据内容:")
for k, v in gguf.metadata.items():
    print(f"  {k}: {v}")

print(f"\nGGUF核心特性:")
print(f"1. 单文件格式: 元数据+权重一体")
print(f"2. mmap支持: 零拷贝加载")
print(f"3. 丰富量化: Q4_0到Q6_K多种选择")
print(f"4. 扩展性: 键值对元数据，易于扩展")

### SafeTensors格式

In [ ]:
class SafeTensorsSimulator:
    """SafeTensors格式模拟"""
    def __init__(self):
        self.tensors = {}

    def add_tensor(self, name, tensor, dtype='F16'):
        self.tensors[name] = {'data': tensor, 'dtype': dtype}

    def simulate_save(self):
        """模拟SafeTensors保存"""
        header = {}
        offset = 0
        for name, info in self.tensors.items():
            dtype_size = 2 if info['dtype'] == 'F16' else 4
            size = info['data'].numel() * dtype_size
            header[name] = {
                'dtype': info['dtype'],
                'shape': list(info['data'].shape),
                'data_offsets': [offset, offset + size]
            }
            offset += size

        header_json = json.dumps(header, separators=(',', ':'))
        header_size = len(header_json.encode())
        total_size = 8 + header_size + offset
        return total_size, header_size

    def simulate_load_mmap(self):
        """模拟mmap加载"""
        load_time = 0.001
        return load_time

st = SafeTensorsSimulator()
for name in ['embed.weight', 'layer.0.q.weight', 'layer.0.k.weight',
             'layer.0.v.weight', 'layer.0.o.weight', 'layer.0.ffn.w1.weight',
             'layer.0.ffn.w2.weight', 'output.weight']:
    st.add_tensor(name, torch.randn(256, 256), dtype='F16')

total_size, header_size = st.simulate_save()

print(f"=== SafeTensors格式 ===")
print(f"张量数量: {len(st.tensors)}")
print(f"头大小: {header_size} bytes")
print(f"总文件大小: {total_size/1024:.2f} KB")

print(f"\nSafeTensors vs PyTorch pickle:")
print(f"  安全性: 无pickle反序列化风险")
print(f"  加载速度: mmap零拷贝，比pickle快3-5x")
print(f"  惰性加载: 可只加载需要的张量")
print(f"  跨框架: HuggingFace标准格式")

print(f"\n=== 模型格式对比总结 ===")
formats = [
    ('GGUF', 'llama.cpp', '量化+mmap', '端侧CPU推理'),
    ('SafeTensors', 'HuggingFace', '安全+mmap', '模型分发/加载'),
    ('ONNX', '跨框架', '标准算子', '推理引擎输入'),
    ('Core ML', 'Apple', 'ANE优化', 'Apple端侧'),
    ('QNN Binary', '高通', 'NPU指令', '骁龙端侧'),
]
print(f"\n{'格式':<15} {'生态':<15} {'特点':<15} {'适用场景':<20}")
print("-" * 65)
for name, eco, feature, scene in formats:
    print(f"{name:<15} {eco:<15} {feature:<15} {scene:<20}")